# Wavelet-Enhanced Hybrid LSTM–SVR for Rainfall Forecasting in Kupang — Revised Experiments

Notebook ini mereproduksi ulang seluruh eksperimen paper secara **bebas kebocoran data (leakage-free)** dan **reproducible**:

* split kronologis Train/Validation/Test (70/15/15) berdasarkan **tanggal target**, identik untuk semua model dan semua `time_steps`;
* `MinMaxScaler` di-fit **hanya** pada bagian train;
* filter Butterworth **kausal** (`lfilter`), bukan `filtfilt`;
* Optuna hanya melihat **validation**; test dipakai **sekali** pada evaluasi akhir;
* seed global tetap, dan semua angka di tabel/gambar dihitung dari variabel (tidak ada angka manual).

Jalankan dari atas ke bawah (`Runtime → Run all`). Semua output disimpan di folder `OUT_DIR`.

## 0. Setup & konfigurasi

Sel ini memasang paket yang belum ada, menetapkan seed global, dan mengumpulkan semua parameter eksperimen di satu tempat. Ubah `QUICK_MODE` menjadi `True` untuk uji cepat (5 trial, 2 seed).

In [ ]:
# ---- Install paket yang belum tersedia (Colab sudah punya sebagian besar) ----
import importlib.util, subprocess, sys
_required = {'pandas': 'pandas', 'numpy': 'numpy', 'matplotlib': 'matplotlib', 'seaborn': 'seaborn',
             'sklearn': 'scikit-learn', 'scipy': 'scipy', 'statsmodels': 'statsmodels',
             'optuna': 'optuna', 'tensorflow': 'tensorflow', 'openpyxl': 'openpyxl'}
_missing = [pip for mod, pip in _required.items() if importlib.util.find_spec(mod) is None]
if _missing:
    print('Installing:', _missing)
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', *_missing])
else:
    print('All required packages are available.')

In [ ]:
# =========================== KONFIGURASI ===========================
import os
from pathlib import Path

QUICK_MODE = False          # True = uji cepat (5 trial, 2 seed); False = run penuh untuk paper

# Path data: dicari di folder kerja, lalu di /content (Colab)
DATA_FILE = 'Data Capstone Fix BGt.xlsx'
DATA_PATH = next((p for p in [Path(DATA_FILE), Path('/content') / DATA_FILE] if p.exists()), Path(DATA_FILE))
OUT_DIR = Path('outputs')

SEED = 42
TRAIN_RATIO, VAL_RATIO, TEST_RATIO = 0.70, 0.15, 0.15

N_TRIALS = 30               # trial Optuna per skenario
EPOCHS = 20
BATCH_SIZE = 32
TS_MIN, TS_MAX = 3, 7       # rentang time_steps (hari)
N_SEEDS_FINAL = 5           # pengulangan seed untuk model berbasis LSTM pada evaluasi akhir

BUTTER_ORDER = 4            # orde filter Butterworth
BUTTER_CUTOFF = 0.2         # cutoff (siklus/hari)
BUTTER_FS = 1.0             # frekuensi sampling (sampel/hari)

# Penanganan kode BMKG
MISSING_CODES = [8888, 9999]   # 8888 = tidak terukur, 9999 = tidak ada data -> NaN
TTU_VALUE = float('nan')       # 'TTU' (tidak terukur) diperlakukan sama seperti 8888 -> NaN.
                               # Notebook lama mengganti TTU dengan 0.1 mm; ubah di sini bila ingin itu.
NEGATIVE_EVAP_TO_NAN = True    # penguapan < 0 mm tidak fisis -> NaN (lalu forward-fill)

# Analisis opsional fase ENSO
USE_ONI = False
ONI_PATH = Path('oni.csv')     # kolom: tahun, bulan, nilai ONI

if QUICK_MODE:
    N_TRIALS, N_SEEDS_FINAL = 5, 2

OUT_DIR.mkdir(parents=True, exist_ok=True)
(OUT_DIR / 'optuna_trials').mkdir(exist_ok=True)

COL_DATE = 'Tanggal'
COL_RAIN = 'Curah Hujan'
COL_HUM = 'Kelembapan Rata-Rata Udara Harian (%)'
COL_SUN = 'Penyinaran Harian (%)'
COL_EVAP = 'Penguapan Harian (mm)'
USED_COLS = [COL_RAIN, COL_HUM, COL_SUN, COL_EVAP]
EN_NAMES = {COL_RAIN: 'Rainfall', COL_HUM: 'Avg Humidity', COL_SUN: 'Sunshine Duration', COL_EVAP: 'Evaporation'}

print(f'QUICK_MODE={QUICK_MODE} | N_TRIALS={N_TRIALS} | N_SEEDS_FINAL={N_SEEDS_FINAL} | DATA_PATH={DATA_PATH}')

In [ ]:
# ---- Seed global & determinisme (harus sebelum TensorFlow di-import) ----
os.environ['PYTHONHASHSEED'] = str(SEED)
os.environ['TF_DETERMINISTIC_OPS'] = '1'
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '2'

import random, time, warnings
import numpy as np
import pandas as pd
import matplotlib
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import seaborn as sns
from scipy.signal import butter, lfilter, lfilter_zi, filtfilt
from sklearn.preprocessing import MinMaxScaler
from sklearn.svm import SVR
from statsmodels.tsa.stattools import adfuller, acf, pacf
from statsmodels.graphics.tsaplots import plot_acf, plot_pacf
import optuna
import tensorflow as tf
from IPython.display import display, Markdown

warnings.filterwarnings('ignore', category=UserWarning)
optuna.logging.set_verbosity(optuna.logging.WARNING)

def set_all_seeds(seed):
    # random, numpy, dan tensorflow sekaligus
    random.seed(seed)
    np.random.seed(seed)
    tf.keras.utils.set_random_seed(seed)

set_all_seeds(SEED)
try:
    tf.config.experimental.enable_op_determinism()
    print('TensorFlow op determinism: ON')
except Exception as e:
    print('TensorFlow op determinism not available:', e)

plt.rcParams.update({'font.size': 12, 'axes.titlesize': 13, 'axes.labelsize': 12,
                     'figure.dpi': 100, 'savefig.bbox': 'tight'})
C_BASE, C_WAVE, C_OBS, C_GREY = '#2a78d6', '#eb6834', '#222222', '#9a9a9a'

print('numpy', np.__version__, '| pandas', pd.__version__, '| tensorflow', tf.__version__, '| optuna', optuna.__version__)

## 1. Load & cleaning

Membaca data BMKG, memastikan urutan tanggal harian lengkap, mengonversi kode 8888/9999 (dan 'TTU') menjadi NaN, lalu imputasi **kausal** (forward fill saja, tanpa `bfill`).

In [ ]:
raw = pd.read_excel(DATA_PATH)
raw[COL_DATE] = pd.to_datetime(raw[COL_DATE])
raw = raw.sort_values(COL_DATE)
print('Rows read:', len(raw), '| columns:', list(raw.columns))

# Tanggal duplikat
n_dup = raw[COL_DATE].duplicated().sum()
print('Duplicated dates:', n_dup)
raw = raw.drop_duplicates(subset=COL_DATE, keep='first').set_index(COL_DATE)

# Reindex ke frekuensi harian penuh -> tanggal yang hilang menjadi baris NaN
full_idx = pd.date_range(raw.index.min(), raw.index.max(), freq='D')
n_missing_dates = len(full_idx) - len(raw)
df = raw.reindex(full_idx)[USED_COLS].copy()
df.index.name = 'date'
print(f'Date range: {df.index.min().date()} – {df.index.max().date()} | days: {len(df)} | missing dates: {n_missing_dates}')

In [ ]:
def to_numeric_bmkg(s):
    # Kode BMKG 8888/9999 (string maupun angka) -> NaN; 'TTU' -> TTU_VALUE
    ttu = s.astype(str).str.strip().str.upper() == 'TTU'
    num = pd.to_numeric(s.where(~ttu), errors='coerce')
    other_nonnumeric = int((num.isna() & s.notna() & ~ttu).sum())
    code_mask = num.isin(MISSING_CODES)
    num[code_mask] = np.nan
    num[ttu] = TTU_VALUE
    return num, {'TTU': int(ttu.sum()), '8888/9999': int(code_mask.sum()), 'other non-numeric': other_nonnumeric}

missing_report = {}
for c in USED_COLS:
    nan_before = int(df[c].isna().sum())
    df[c], info = to_numeric_bmkg(df[c])
    info['NaN from reindex/raw'] = nan_before
    missing_report[c] = info

if NEGATIVE_EVAP_TO_NAN:
    neg = df[COL_EVAP] < 0
    missing_report[COL_EVAP]['negative (invalid)'] = int(neg.sum())
    if neg.any():
        print('Negative evaporation values set to NaN:', df.loc[neg, COL_EVAP].to_dict())
    df.loc[neg, COL_EVAP] = np.nan

missing_tbl = pd.DataFrame(missing_report).T.fillna(0).astype(int)
missing_tbl['total NaN after conversion'] = df.isna().sum()
missing_tbl.index = [EN_NAMES[c] for c in missing_tbl.index]
print('Missing values per column (before imputation):')
display(missing_tbl)

In [ ]:
# Imputasi KAUSAL: forward fill saja (nilai hari t hanya diisi dari hari <= t). TIDAK memakai bfill.
df = df.ffill()

# Jika awal seri masih NaN (tidak ada nilai sebelumnya untuk di-ffill), baris awal tersebut di-drop
# agar tidak ada informasi masa depan yang dipakai untuk mengisi masa lalu.
first_valid = df.dropna().index.min()
n_dropped_head = int((df.index < first_valid).sum())
df = df.loc[first_valid:]
assert df.isna().sum().sum() == 0
print(f'Leading rows dropped (still NaN after ffill): {n_dropped_head}')
print(f'Final clean series: {df.index.min().date()} – {df.index.max().date()} | N = {len(df)} days')

desc = df.rename(columns=EN_NAMES).describe().T
display(desc.round(3))
desc.to_csv(OUT_DIR / 'descriptive_statistics.csv')

PCT_DRY_DAYS = 100 * (df[COL_RAIN] == 0).mean()
print(f'Days without rain (rainfall = 0 mm): {PCT_DRY_DAYS:.2f}%')

## 2. Exploratory data analysis (Gambar 2, 3, 4)

Time series dan distribusi curah hujan, korelasi Pearson hanya untuk 4 variabel yang dipakai model, ACF/PACF (36 lag), serta uji Augmented Dickey–Fuller. Statistik EDA ini deskriptif dan tidak dipakai untuk memilih hiperparameter.

In [ ]:
fig, ax = plt.subplots(figsize=(14, 5))
ax.plot(df.index, df[COL_RAIN], color=C_BASE, lw=0.9)
ax.set(title='Daily Rainfall in Kupang (2018–2022)', xlabel='Date', ylabel='Rainfall (mm)')
ax.xaxis.set_major_locator(mdates.YearLocator()); ax.xaxis.set_major_formatter(mdates.DateFormatter('%Y'))
ax.grid(alpha=0.3)
fig.savefig(OUT_DIR / 'fig2_rainfall_timeseries.png', dpi=300); plt.show()

fig, ax = plt.subplots(figsize=(6, 6))
sns.violinplot(y=df[COL_RAIN], color='#86b6ef', ax=ax, cut=0)
ax.set(title='Distribution of Daily Rainfall', ylabel='Rainfall (mm)')
fig.savefig(OUT_DIR / 'fig3_rainfall_violin.png', dpi=300); plt.show()

corr = df[USED_COLS].rename(columns=EN_NAMES).corr(method='pearson')
fig, ax = plt.subplots(figsize=(7.5, 6))
sns.heatmap(corr, annot=True, fmt='.2f', cmap='RdBu_r', vmin=-1, vmax=1, center=0, square=True, ax=ax)
ax.set_title('Pearson Correlation of Rainfall and Meteorological Variables')
fig.savefig(OUT_DIR / 'fig4_correlation_heatmap.png', dpi=300); plt.show()
display(corr.round(3))

In [ ]:
N_LAGS = 36
rain = df[COL_RAIN].values

fig, axes = plt.subplots(1, 2, figsize=(15, 5))
plot_acf(rain, lags=N_LAGS, ax=axes[0]); axes[0].set_title('Autocorrelation Function (ACF) of Rainfall')
plot_pacf(rain, lags=N_LAGS, ax=axes[1], method='ywm'); axes[1].set_title('Partial Autocorrelation Function (PACF) of Rainfall')
for a in axes: a.set_xlabel('Lag (days)')
fig.savefig(OUT_DIR / 'fig_acf_pacf.png', dpi=300); plt.show()

# Lag signifikan: interval kepercayaan 95% tidak memuat 0
acf_vals, acf_ci = acf(rain, nlags=N_LAGS, alpha=0.05)
pacf_vals, pacf_ci = pacf(rain, nlags=N_LAGS, alpha=0.05, method='ywm')
def significant_lags(vals, ci):
    lo, hi = ci[:, 0] - vals, ci[:, 1] - vals   # setengah lebar interval di sekitar 0
    return [k for k in range(1, len(vals)) if vals[k] > hi[k] or vals[k] < lo[k]]
SIG_ACF = significant_lags(acf_vals, acf_ci)
SIG_PACF = significant_lags(pacf_vals, pacf_ci)
print('Significant ACF lags (95%): ', SIG_ACF)
print('Significant PACF lags (95%):', SIG_PACF)

In [ ]:
adf_res = adfuller(rain, autolag='AIC')
ADF_STAT, ADF_P, ADF_USEDLAG, ADF_NOBS, ADF_CRIT = adf_res[0], adf_res[1], adf_res[2], adf_res[3], adf_res[4]
ADF_CONCLUSION = ('reject H0 (unit root) at the 5% level -> the series is stationary'
                  if ADF_P < 0.05 else 'fail to reject H0 -> the series is non-stationary')
print('Augmented Dickey–Fuller test on daily rainfall (autolag = AIC)')
print(f'  ADF statistic : {ADF_STAT:.4f}')
print(f'  p-value       : {ADF_P:.3e}')
print(f'  lags used     : {ADF_USEDLAG} | observations: {ADF_NOBS}')
for k, v in ADF_CRIT.items():
    print(f'  critical {k:>3} : {v:.4f}')
print('  conclusion    :', ADF_CONCLUSION)

## 3. Split kronologis (identik untuk semua model)

Sampel didefinisikan per **tanggal target**. Tanggal target dimulai dari indeks `TS_MAX` sehingga setiap `time_steps` (3–7) punya input yang lengkap, dan split 70/15/15 diterapkan pada daftar tanggal target ini. Jendela input boleh melintasi batas split (memakai data masa lalu), tetapi tanggal target test identik untuk semua model.

In [ ]:
N = len(df)
dates = df.index
target_idx_all = np.arange(TS_MAX, N)          # indeks baris yang menjadi target (hari t)
n_targets = len(target_idx_all)
n_train = int(round(TRAIN_RATIO * n_targets))
n_val = int(round(VAL_RATIO * n_targets))
idx_train = target_idx_all[:n_train]
idx_val = target_idx_all[n_train:n_train + n_val]
idx_test = target_idx_all[n_train + n_val:]
idx_trval = np.concatenate([idx_train, idx_val])

SPLITS = {'Train': idx_train, 'Validation': idx_val, 'Test': idx_test}
split_tbl = pd.DataFrame({k: {'start': dates[v[0]].date(), 'end': dates[v[-1]].date(), 'n_samples': len(v),
                              'dry days (%)': round(100 * (df[COL_RAIN].values[v] == 0).mean(), 2)}
                          for k, v in SPLITS.items()}).T
display(split_tbl)
split_tbl.to_csv(OUT_DIR / 'split_ranges.csv')
TEST_START, TEST_END = dates[idx_test[0]].date(), dates[idx_test[-1]].date()

# MinMaxScaler di-fit HANYA pada baris yang tersedia untuk train (s.d. tanggal target train terakhir)
train_rows = slice(0, idx_train[-1] + 1)
scaler = MinMaxScaler()
scaler.fit(df[USED_COLS].values[train_rows])
scaled = scaler.transform(df[USED_COLS].values)     # val/test bisa sedikit di luar [0, 1] — itu wajar
RAIN_MIN, RAIN_RANGE = scaler.data_min_[0], scaler.data_range_[0]
print(f'Scaler fitted on {idx_train[-1] + 1} rows ({dates[0].date()} – {dates[idx_train[-1]].date()}); '
      f'rainfall train min/max = {RAIN_MIN:.1f}/{RAIN_MIN + RAIN_RANGE:.1f} mm')

y_scaled_all = scaled[:, 0]          # target ter-scale
y_mm_all = df[COL_RAIN].values       # target asli (mm)
to_mm = lambda z: np.asarray(z) * RAIN_RANGE + RAIN_MIN

## 4. Pseudo-wavelet (Butterworth kausal)

Curah hujan ter-scale didekomposisi dengan low-pass Butterworth orde 4 (cutoff 0.2 siklus/hari) memakai `scipy.signal.lfilter` yang **kausal**: Approximation `A[t]` hanya bergantung pada `x[≤t]`, dan Detail `D = x − A`. Fitur skenario wavelet: `[A, D, humidity, sunshine, evaporation]`; skenario base: `[rainfall, humidity, sunshine, evaporation]`. Target tetap curah hujan ter-scale hari t (diprediksi dari jendela hari t−ts … t−1).

In [ ]:
B_BUTTER, A_BUTTER = butter(N=BUTTER_ORDER, Wn=BUTTER_CUTOFF, btype='low', fs=BUTTER_FS)

def causal_decompose(x):
    # Kondisi awal = steady state pada x[0] (hanya memakai sampel pertama) -> tetap kausal, transien awal kecil
    zi = lfilter_zi(B_BUTTER, A_BUTTER) * x[0]
    approx, _ = lfilter(B_BUTTER, A_BUTTER, x, zi=zi)
    return approx, x - approx

def noncausal_decompose(x):
    # HANYA UNTUK DIAGNOSIS (ablation): filtfilt memakai sampel masa depan
    approx = filtfilt(B_BUTTER, A_BUTTER, x)
    return approx, x - approx

def build_features(wavelet, filt='causal'):
    if not wavelet:
        return scaled.copy()                               # [rain, hum, sun, evap]
    A, D = (causal_decompose if filt == 'causal' else noncausal_decompose)(y_scaled_all)
    return np.column_stack([A, D, scaled[:, 1:]])          # [A, D, hum, sun, evap]

FEATURES = {False: build_features(False), True: build_features(True, 'causal')}

# Cek kausalitas: memotong seri di titik mana pun tidak mengubah A sebelum titik itu
_cut = idx_train[-1] + 1
_A_full, _ = causal_decompose(y_scaled_all)
_A_cut, _ = causal_decompose(y_scaled_all[:_cut])
assert np.allclose(_A_full[:_cut], _A_cut), 'lfilter output must not depend on future samples'
print('Causality check passed: A[t] computed on the truncated series equals A[t] on the full series.')

def make_windows(F, target_idx, ts):
    # Input: baris t-ts ... t-1 ; target: curah hujan ter-scale pada baris t
    X = np.stack([F[t - ts:t] for t in target_idx])
    return X, y_scaled_all[target_idx]

In [ ]:
n_demo = 180
A_c, D_c = causal_decompose(y_scaled_all)
A_nc, _ = noncausal_decompose(y_scaled_all)
fig, axes = plt.subplots(2, 1, figsize=(15, 8), sharex=True)
axes[0].plot(dates[:n_demo], y_scaled_all[:n_demo], color=C_GREY, lw=1.2, label='Rainfall (scaled)')
axes[0].plot(dates[:n_demo], A_c[:n_demo], color=C_BASE, lw=2, label='Approximation A (causal lfilter)')
axes[0].plot(dates[:n_demo], A_nc[:n_demo], color=C_WAVE, lw=1.5, ls='--', label='Approximation (non-causal filtfilt, diagnostic only)')
axes[0].set(ylabel='Scaled value', title=f'Causal Butterworth decomposition (order {BUTTER_ORDER}, cutoff {BUTTER_CUTOFF} cycles/day) — first {n_demo} days')
axes[0].legend(loc='upper right'); axes[0].grid(alpha=0.3)
axes[1].plot(dates[:n_demo], D_c[:n_demo], color=C_OBS, lw=1.2, label='Detail D = x − A')
axes[1].axhline(0, color=C_GREY, lw=0.8)
axes[1].set(ylabel='Scaled value', xlabel='Date'); axes[1].legend(loc='upper right'); axes[1].grid(alpha=0.3)
fig.savefig(OUT_DIR / 'fig_wavelet_decomposition.png', dpi=300); plt.show()

## 5. Optimasi hiperparameter dengan Optuna (validation only)

Setiap skenario dioptimasi `N_TRIALS` kali dengan TPE sampler ber-seed. Model dilatih pada **train** dan dinilai dengan RMSE (skala 0–1) pada **validation**; test set tidak disentuh di bagian ini. Hybrid LSTM–SVR: LSTM (LSTM + Dense(1)) dilatih di train, output layer LSTM (hidden state terakhir) dipakai sebagai fitur untuk SVR (RBF).

In [ ]:
SCENARIOS = {
    'Base SVR':      {'model': 'svr',      'wavelet': False},
    'W-SVR':         {'model': 'svr',      'wavelet': True},
    'Base LSTM':     {'model': 'lstm',     'wavelet': False},
    'W-LSTM':        {'model': 'lstm',     'wavelet': True},
    'Base LSTM-SVR': {'model': 'lstm_svr', 'wavelet': False},
    'W-LSTM-SVR':    {'model': 'lstm_svr', 'wavelet': True},
}

rmse = lambda a, b: float(np.sqrt(np.mean((np.asarray(a) - np.asarray(b)) ** 2)))
mae = lambda a, b: float(np.mean(np.abs(np.asarray(a) - np.asarray(b))))

def build_lstm(ts, n_feat, units, lr):
    inp = tf.keras.Input(shape=(ts, n_feat))
    h = tf.keras.layers.LSTM(units, name='lstm')(inp)
    out = tf.keras.layers.Dense(1, name='dense_head')(h)
    model = tf.keras.Model(inp, out)
    model.compile(optimizer=tf.keras.optimizers.Adam(learning_rate=lr), loss='mse')
    return model

def fit_predict(kind, params, F, fit_idx, eval_idx_list, seed):
    # Melatih satu model pada fit_idx dan mengembalikan prediksi (skala 0-1) untuk tiap array di eval_idx_list
    ts = params['time_steps']
    X_fit, y_fit = make_windows(F, fit_idx, ts)
    X_evals = [make_windows(F, ix, ts)[0] for ix in eval_idx_list]
    if kind == 'svr':   # deterministik
        svr = SVR(kernel='rbf', C=params['C'], gamma=params['gamma'], epsilon=params['epsilon'])
        svr.fit(X_fit.reshape(len(X_fit), -1), y_fit)
        return [svr.predict(X.reshape(len(X), -1)) for X in X_evals]
    tf.keras.backend.clear_session()
    set_all_seeds(seed)
    net = build_lstm(ts, F.shape[1], params['lstm_units'], params['learning_rate'])
    net.fit(X_fit, y_fit, epochs=EPOCHS, batch_size=BATCH_SIZE, verbose=0, shuffle=True)
    if kind == 'lstm':
        return [net(X, training=False).numpy().ravel() for X in X_evals]
    # Hybrid: hidden state LSTM sebagai fitur -> SVR
    extractor = tf.keras.Model(net.input, net.get_layer('lstm').output)
    H_fit = extractor(X_fit, training=False).numpy()
    svr = SVR(kernel='rbf', C=params['C'], gamma=params['gamma'], epsilon=params['epsilon'])
    svr.fit(H_fit, y_fit)
    return [svr.predict(extractor(X, training=False).numpy()) for X in X_evals]

def suggest_params(trial, kind):
    p = {'time_steps': trial.suggest_int('time_steps', TS_MIN, TS_MAX)}
    if kind in ('lstm', 'lstm_svr'):
        p['lstm_units'] = trial.suggest_categorical('lstm_units', [32, 64, 96, 128])
        p['learning_rate'] = trial.suggest_categorical('learning_rate', [1e-2, 1e-3, 1e-4])
    if kind in ('svr', 'lstm_svr'):
        p['C'] = trial.suggest_float('C', 0.1, 100, log=True)
        p['gamma'] = trial.suggest_float('gamma', 1e-3, 1.0, log=True)
        p['epsilon'] = trial.suggest_float('epsilon', 1e-3, 0.1, log=True)
    return p

def make_objective(kind, wavelet):
    F = FEATURES[wavelet]
    def objective(trial):
        p = suggest_params(trial, kind)
        # Latih di TRAIN, nilai di VALIDATION (test tidak dipakai)
        (pred_val,) = fit_predict(kind, p, F, idx_train, [idx_val], seed=SEED)
        return rmse(y_scaled_all[idx_val], pred_val)
    return objective

In [ ]:
studies, best_params = {}, {}
t_opt = time.time()
for name, cfg in SCENARIOS.items():
    t0 = time.time()
    study = optuna.create_study(direction='minimize', study_name=name,
                                sampler=optuna.samplers.TPESampler(seed=SEED))
    study.optimize(make_objective(cfg['model'], cfg['wavelet']), n_trials=N_TRIALS)
    studies[name] = study
    best_params[name] = dict(study.best_params)
    trials_df = study.trials_dataframe()
    trials_df.insert(0, 'scenario', name)
    trials_df.to_csv(OUT_DIR / 'optuna_trials' / f"trials_{name.replace(' ', '_')}.csv", index=False)
    print(f'{name:<14} best val RMSE (scaled) = {study.best_value:.5f} | {best_params[name]} | {time.time() - t0:.0f}s')
print(f'Total Optuna time: {(time.time() - t_opt) / 60:.1f} min')

all_trials = pd.concat([pd.read_csv(f) for f in sorted((OUT_DIR / 'optuna_trials').glob('trials_*.csv'))], ignore_index=True)
all_trials.to_csv(OUT_DIR / 'optuna_trials_all.csv', index=False)

best_tbl = pd.DataFrame(best_params).T
best_tbl['best val RMSE (scaled)'] = [studies[n].best_value for n in best_tbl.index]
display(best_tbl)

## 6. Evaluasi akhir pada TEST (satu kali)

Setiap skenario dilatih ulang dengan parameter terbaik pada **train + validation**, lalu dievaluasi sekali di test. Model berbasis LSTM diulang dengan `N_SEEDS_FINAL` seed (mean ± SD); SVR deterministik cukup sekali. Prediksi dikembalikan ke mm, nilai negatif di-clip ke 0 (metrik utama setelah clipping).

In [ ]:
FINAL_SEEDS = [SEED + k for k in range(N_SEEDS_FINAL)]
y_test_mm = y_mm_all[idx_test]
test_dates = dates[idx_test]

def evaluate_runs(preds_scaled, y_true_mm):
    # preds_scaled: list prediksi (satu per seed). Return metrik per run.
    rows = []
    for p in preds_scaled:
        p_mm_raw = to_mm(p)
        p_mm = np.clip(p_mm_raw, 0, None)
        rows.append({'RMSE_mm': rmse(y_true_mm, p_mm), 'MAE_mm': mae(y_true_mm, p_mm),
                     'RMSE_scaled': rmse(y_true_mm, p_mm) / RAIN_RANGE,
                     'RMSE_mm_unclipped': rmse(y_true_mm, p_mm_raw), 'MAE_mm_unclipped': mae(y_true_mm, p_mm_raw)})
    return pd.DataFrame(rows)

final_runs, final_preds_mm, final_val_preds = {}, {}, {}
for name, cfg in SCENARIOS.items():
    t0 = time.time()
    seeds = [SEED] if cfg['model'] == 'svr' else FINAL_SEEDS
    preds = [fit_predict(cfg['model'], best_params[name], FEATURES[cfg['wavelet']], idx_trval, [idx_test], seed=s)[0]
             for s in seeds]
    runs = evaluate_runs(preds, y_test_mm)
    runs.insert(0, 'seed', seeds)
    final_runs[name] = runs
    final_preds_mm[name] = [np.clip(to_mm(p), 0, None) for p in preds]
    sd_txt = f' ± {runs.RMSE_mm.std(ddof=1):.3f}' if len(runs) > 1 else ''
    print(f"{name:<14} RMSE = {runs.RMSE_mm.mean():.3f}{sd_txt} mm | MAE = {runs.MAE_mm.mean():.3f} mm "
          f"| runs = {len(runs)} | {time.time() - t0:.0f}s")

pd.concat({k: v for k, v in final_runs.items()}, names=['Model', 'run']).to_csv(OUT_DIR / 'final_test_runs_per_seed.csv')

In [ ]:
# ---- Baseline (tanpa pelatihan): persistensi dan rata-rata train ----
persist_mm = y_mm_all[idx_test - 1]                         # prediksi hari t = curah hujan hari t-1
train_mean_mm = np.full_like(y_test_mm, y_mm_all[idx_train].mean(), dtype=float)
BASELINES = {'Persistence (t+1 = t)': persist_mm, 'Train mean': train_mean_mm}
baseline_runs = {k: pd.DataFrame([{'seed': np.nan, 'RMSE_mm': rmse(y_test_mm, v), 'MAE_mm': mae(y_test_mm, v),
                                    'RMSE_scaled': rmse(y_test_mm, v) / RAIN_RANGE,
                                    'RMSE_mm_unclipped': rmse(y_test_mm, v), 'MAE_mm_unclipped': mae(y_test_mm, v)}])
                 for k, v in BASELINES.items()}

def summarize(name, runs, ts):
    n = len(runs)
    sd = lambda c: runs[c].std(ddof=1) if n > 1 else np.nan
    return {'Model': name, 'time_steps': ts, 'n_runs': n,
            'RMSE (mm)': runs.RMSE_mm.mean(), 'RMSE SD (mm)': sd('RMSE_mm'),
            'MAE (mm)': runs.MAE_mm.mean(), 'MAE SD (mm)': sd('MAE_mm'),
            'RMSE scaled': runs.RMSE_scaled.mean(), 'RMSE scaled SD': sd('RMSE_scaled'),
            'RMSE (mm) before clipping': runs.RMSE_mm_unclipped.mean()}

results = pd.DataFrame([summarize(n, final_runs[n], best_params[n]['time_steps']) for n in SCENARIOS] +
                       [summarize(n, r, np.nan) for n, r in baseline_runs.items()])

def pm(m, s, d=3):
    return f'{m:.{d}f}' if pd.isna(s) else f'{m:.{d}f} ± {s:.{d}f}'
results['RMSE (mm) mean ± SD'] = [pm(m, s) for m, s in zip(results['RMSE (mm)'], results['RMSE SD (mm)'])]
results['MAE (mm) mean ± SD'] = [pm(m, s) for m, s in zip(results['MAE (mm)'], results['MAE SD (mm)'])]
results['RMSE scaled mean ± SD'] = [pm(m, s, 4) for m, s in zip(results['RMSE scaled'], results['RMSE scaled SD'])]
results.to_csv(OUT_DIR / 'results_table.csv', index=False)
display(results)

In [ ]:
# ---- Penurunan RMSE akibat wavelet: (RMSE_base - RMSE_wavelet) / RMSE_base x 100 (RMSE mm, mean) ----
res_idx = results.set_index('Model')
PAIRS = [('SVR', 'Base SVR', 'W-SVR'), ('LSTM', 'Base LSTM', 'W-LSTM'), ('LSTM-SVR', 'Base LSTM-SVR', 'W-LSTM-SVR')]
reduction = pd.DataFrame([{'Model family': fam, 'RMSE base (mm)': res_idx.loc[b, 'RMSE (mm)'],
                           'RMSE wavelet (mm)': res_idx.loc[w, 'RMSE (mm)'],
                           'RMSE reduction (%)': 100 * (res_idx.loc[b, 'RMSE (mm)'] - res_idx.loc[w, 'RMSE (mm)']) / res_idx.loc[b, 'RMSE (mm)']}
                          for fam, b, w in PAIRS])
reduction.to_csv(OUT_DIR / 'wavelet_rmse_reduction.csv', index=False)
display(reduction.round(3))

# Perbandingan terhadap baseline persistensi
PERSIST_RMSE = res_idx.loc['Persistence (t+1 = t)', 'RMSE (mm)']
results['Improvement vs persistence (%)'] = 100 * (PERSIST_RMSE - results['RMSE (mm)']) / PERSIST_RMSE
display(results[['Model', 'RMSE (mm) mean ± SD', 'Improvement vs persistence (%)']].round(2))
results.to_csv(OUT_DIR / 'results_table.csv', index=False)

In [ ]:
# ---- Tabel II siap tempel (Markdown & LaTeX), dibuat dari DataFrame tanpa dependensi tambahan ----
tab2 = results[['Model', 'time_steps', 'RMSE (mm) mean ± SD', 'MAE (mm) mean ± SD', 'RMSE scaled mean ± SD']].copy()
tab2['time_steps'] = tab2['time_steps'].map(lambda v: '–' if pd.isna(v) else str(int(v)))
tab2.columns = ['Model', 'Time steps', 'RMSE (mm)', 'MAE (mm)', 'RMSE (scaled)']

def to_markdown(t):
    lines = ['| ' + ' | '.join(t.columns) + ' |', '|' + '|'.join(['---'] * t.shape[1]) + '|']
    lines += ['| ' + ' | '.join(map(str, r)) + ' |' for r in t.values]
    return '\n'.join(lines)

def to_latex(t, caption, label):
    esc = lambda s: str(s).replace('±', r'$\pm$').replace('–', '--').replace('%', r'\%').replace('_', r'\_')
    body = [' & '.join(esc(v) for v in r) + r' \\' for r in t.values]
    return '\n'.join([r'\begin{table}[htbp]', r'\centering', rf'\caption{{{caption}}}', rf'\label{{{label}}}',
                      r'\begin{tabular}{' + 'l' + 'c' * (t.shape[1] - 1) + '}', r'\hline',
                      ' & '.join(esc(c) for c in t.columns) + r' \\', r'\hline', *body, r'\hline',
                      r'\end{tabular}', r'\end{table}'])

note = (f'Test period {TEST_START} to {TEST_END} (n = {len(idx_test)} days). LSTM-based models: mean ± SD over '
        f'{N_SEEDS_FINAL} seeds; SVR and baselines are deterministic.')
TAB2_MD = to_markdown(tab2) + '\n\n' + note
TAB2_TEX = to_latex(tab2, 'Forecasting performance on the test set. ' + note, 'tab:results')
(OUT_DIR / 'table2_results.md').write_text(TAB2_MD)
(OUT_DIR / 'table2_results.tex').write_text(TAB2_TEX)
print(TAB2_MD); print(); print(TAB2_TEX)

In [ ]:
# ---- Tabel parameter final (Methods) ----
param_rows = []
for name, cfg in SCENARIOS.items():
    p = best_params[name]
    lstm_based = cfg['model'] != 'svr'
    svr_based = cfg['model'] != 'lstm'
    param_rows.append({
        'Model': name, 'time_steps': p['time_steps'],
        'LSTM units': p.get('lstm_units', '–'), 'learning rate': p.get('learning_rate', '–'),
        'C': round(p['C'], 4) if svr_based else '–', 'gamma': round(p['gamma'], 5) if svr_based else '–',
        'epsilon': round(p['epsilon'], 5) if svr_based else '–',
        'epochs': EPOCHS if lstm_based else '–', 'batch size': BATCH_SIZE if lstm_based else '–',
        'optimizer': 'Adam' if lstm_based else '–', 'loss': 'MSE' if lstm_based else 'ε-insensitive',
        'kernel': 'RBF' if svr_based else '–',
        'Optuna trials': N_TRIALS, 'Optuna objective': 'validation RMSE (scaled)',
        'seed': SEED, 'final seeds': N_SEEDS_FINAL if lstm_based else 1,
    })
best_params_table = pd.DataFrame(param_rows)
best_params_table.to_csv(OUT_DIR / 'best_params_table.csv', index=False)
display(best_params_table)
print(to_markdown(best_params_table.astype(str)))

### 6b. Ablation kausalitas: `filtfilt` vs `lfilter` (DIAGNOSIS SAJA)

> ⚠️ **Hasil `filtfilt` di sel ini BUKAN hasil utama.** `filtfilt` memproses seri maju-mundur sehingga fitur pada hari t memakai curah hujan hari t+1, t+2, … (termasuk target). Sel ini hanya mengukur seberapa besar "perbaikan" pada notebook lama yang sebenarnya berasal dari kebocoran masa depan. Hiperparameter yang dipakai sama dengan versi kausal (hasil Optuna di atas).

In [ ]:
FEAT_FILTFILT = build_features(True, 'noncausal')   # dihitung pada seluruh seri, seperti notebook lama
abl_rows = []
for name in ['W-SVR', 'W-LSTM']:
    cfg = SCENARIOS[name]
    seeds = [SEED] if cfg['model'] == 'svr' else FINAL_SEEDS
    for filt, F in [('lfilter (causal)', FEATURES[True]), ('filtfilt (non-causal, diagnostic)', FEAT_FILTFILT)]:
        val_r, test_r = [], []
        for s in seeds:
            # validation: latih di train ; test: latih di train+val (protokol sama dengan evaluasi akhir)
            (pv,) = fit_predict(cfg['model'], best_params[name], F, idx_train, [idx_val], seed=s)
            (pt,) = fit_predict(cfg['model'], best_params[name], F, idx_trval, [idx_test], seed=s)
            val_r.append(rmse(y_mm_all[idx_val], np.clip(to_mm(pv), 0, None)))
            test_r.append(rmse(y_test_mm, np.clip(to_mm(pt), 0, None)))
        abl_rows.append({'Model': name, 'Filter': filt, 'n_runs': len(seeds),
                         'Val RMSE (mm)': np.mean(val_r), 'Test RMSE (mm)': np.mean(test_r),
                         'Test RMSE SD (mm)': np.std(test_r, ddof=1) if len(test_r) > 1 else np.nan})
ablation_causality = pd.DataFrame(abl_rows)
for name in ['W-SVR', 'W-LSTM']:
    sub = ablation_causality[ablation_causality.Model == name].set_index('Filter')['Test RMSE (mm)']
    is_ff = (ablation_causality.Model == name) & ablation_causality.Filter.str.startswith('filtfilt')
    ablation_causality.loc[is_ff, 'Apparent RMSE gain from leakage (%)'] = 100 * (sub.iloc[0] - sub.iloc[1]) / sub.iloc[0]
ablation_causality.to_csv(OUT_DIR / 'ablation_causality_filtfilt_vs_lfilter.csv', index=False)
display(ablation_causality.round(3))

## 7. Analisis tambahan untuk pertanyaan reviewer

**7a.** Mengapa W-LSTM bisa lebih baik dari W-LSTM-SVR: satu LSTM (parameter terbaik W-LSTM-SVR, dilatih di train) dipakai bersama, lalu head Dense bawaan dibandingkan dengan head SVR pada grid kecil epsilon × gamma (C tetap = nilai terbaik). Semua dinilai di **validation**.

In [ ]:
p_h = best_params['W-LSTM-SVR']
F_w = FEATURES[True]
ts_h = p_h['time_steps']
X_tr_h, y_tr_h = make_windows(F_w, idx_train, ts_h)
X_va_h, _ = make_windows(F_w, idx_val, ts_h)
y_va_mm = y_mm_all[idx_val]

tf.keras.backend.clear_session(); set_all_seeds(SEED)
net_h = build_lstm(ts_h, F_w.shape[1], p_h['lstm_units'], p_h['learning_rate'])
net_h.fit(X_tr_h, y_tr_h, epochs=EPOCHS, batch_size=BATCH_SIZE, verbose=0)
extractor_h = tf.keras.Model(net_h.input, net_h.get_layer('lstm').output)
H_tr, H_va = extractor_h(X_tr_h, training=False).numpy(), extractor_h(X_va_h, training=False).numpy()

dense_val_rmse = rmse(y_va_mm, np.clip(to_mm(net_h(X_va_h, training=False).numpy().ravel()), 0, None))
EPS_GRID = [0.001, 0.005, 0.01, 0.05, 0.1]
GAMMA_GRID = [0.001, 0.01, 0.1, 1.0, p_h['gamma']]
grid_rows = []
for eps in EPS_GRID:
    for g in sorted(set(GAMMA_GRID)):
        svr = SVR(kernel='rbf', C=p_h['C'], gamma=g, epsilon=eps).fit(H_tr, y_tr_h)
        pred = np.clip(to_mm(svr.predict(H_va)), 0, None)
        grid_rows.append({'epsilon': eps, 'gamma': round(g, 5), 'Val RMSE (mm)': rmse(y_va_mm, pred),
                          'Val MAE (mm)': mae(y_va_mm, pred),
                          'support vectors (%)': 100 * len(svr.support_) / len(y_tr_h),
                          'max predicted (mm)': pred.max()})
svr_grid = pd.DataFrame(grid_rows).sort_values('Val RMSE (mm)')
svr_grid.to_csv(OUT_DIR / 'ablation_wlstmsvr_head_grid.csv', index=False)

# W-LSTM (parameter terbaiknya sendiri) di validation, seed yang sama
(pv_wlstm,) = fit_predict('lstm', best_params['W-LSTM'], F_w, idx_train, [idx_val], seed=SEED)
wlstm_val_rmse = rmse(y_va_mm, np.clip(to_mm(pv_wlstm), 0, None))
display(svr_grid.round(4))

In [ ]:
best_g = svr_grid.iloc[0]
worst_g = svr_grid.iloc[-1]
summary_7a = f'''
**Ringkasan 7a (angka dihitung otomatis, validation set {dates[idx_val[0]].date()} – {dates[idx_val[-1]].date()}, n = {len(idx_val)}):**

* LSTM bersama: time_steps = {ts_h}, units = {p_h['lstm_units']}, learning rate = {p_h['learning_rate']}, seed = {SEED}.
* Head **Dense** bawaan LSTM tersebut: Val RMSE = **{dense_val_rmse:.3f} mm**.
* Head **SVR** terbaik pada grid (ε = {best_g['epsilon']}, γ = {best_g['gamma']}, C = {p_h['C']:.3f}): Val RMSE = **{best_g['Val RMSE (mm)']:.3f} mm** (support vectors {best_g['support vectors (%)']:.1f}% dari sampel train, prediksi maksimum {best_g['max predicted (mm)']:.1f} mm).
* Head SVR terburuk pada grid (ε = {worst_g['epsilon']}, γ = {worst_g['gamma']}): Val RMSE = {worst_g['Val RMSE (mm)']:.3f} mm → rentang sensitivitas head SVR = {worst_g['Val RMSE (mm)'] - best_g['Val RMSE (mm)']:.3f} mm.
* Selisih head terbaik SVR − Dense = {best_g['Val RMSE (mm)'] - dense_val_rmse:+.3f} mm (positif = SVR lebih buruk).
* W-LSTM dengan parameter terbaiknya sendiri (seed sama): Val RMSE = {wlstm_val_rmse:.3f} mm.
* Curah hujan maksimum observasi di validation = {y_va_mm.max():.1f} mm; proporsi hari kering di train = {100 * (y_mm_all[idx_train] == 0).mean():.1f}%.
'''
display(Markdown(summary_7a))

**7b.** Analisis error berdasarkan jenis hari pada test set: hari tanpa hujan (0 mm), hari hujan (> 0 mm), dan hari hujan lebat (> persentil 90 curah hujan seluruh hari di periode train). Untuk model berbasis LSTM, metrik dirata-ratakan atas seed.

In [ ]:
HEAVY_THR = float(np.percentile(y_mm_all[idx_train], 90))
GROUPS = {'Dry (0 mm)': y_test_mm == 0, 'Rainy (> 0 mm)': y_test_mm > 0, f'Heavy (> P90 = {HEAVY_THR:.1f} mm)': y_test_mm > HEAVY_THR,
          'All test days': np.ones_like(y_test_mm, dtype=bool)}
all_preds_mm = {**final_preds_mm, **{k: [v] for k, v in BASELINES.items()}}
rows = []
for model, plist in all_preds_mm.items():
    for g, m in GROUPS.items():
        if m.sum() == 0:
            continue
        r = [rmse(y_test_mm[m], p[m]) for p in plist]
        a = [mae(y_test_mm[m], p[m]) for p in plist]
        rows.append({'Model': model, 'Day type': g, 'n_days': int(m.sum()),
                     'RMSE (mm)': np.mean(r), 'MAE (mm)': np.mean(a),
                     'Mean bias (mm)': np.mean([np.mean(p[m] - y_test_mm[m]) for p in plist])})
error_by_daytype = pd.DataFrame(rows)
error_by_daytype.to_csv(OUT_DIR / 'error_by_daytype.csv', index=False)
print(f'Heavy-rain threshold (P90 of train rainfall, all days) = {HEAVY_THR:.2f} mm')
display(error_by_daytype.pivot_table(index='Model', columns='Day type', values='RMSE (mm)', sort=False).round(3))
display(error_by_daytype.pivot_table(index='Model', columns='Day type', values='MAE (mm)', sort=False).round(3))

**7c. (Opsional)** Fase ENSO berdasarkan ONI. Aktifkan dengan `USE_ONI = True` dan sediakan file di `ONI_PATH` (kolom: tahun, bulan, nilai ONI). El Niño: ONI ≥ 0.5; La Niña: ONI ≤ −0.5; selain itu Netral.

In [ ]:
oni_phase_counts, error_by_enso = None, None
if USE_ONI and Path(ONI_PATH).exists():
    oni = pd.read_csv(ONI_PATH) if str(ONI_PATH).endswith('.csv') else pd.read_excel(ONI_PATH)
    oni.columns = ['year', 'month', 'oni'] + list(oni.columns[3:])
    oni_map = {(int(y), int(m)): float(v) for y, m, v in oni[['year', 'month', 'oni']].values}
    oni_daily = np.array([oni_map.get((d.year, d.month), np.nan) for d in dates])
    phase = np.where(oni_daily >= 0.5, 'El Niño', np.where(oni_daily <= -0.5, 'La Niña',
                     np.where(np.isnan(oni_daily), 'No ONI', 'Neutral')))
    oni_phase_counts = pd.DataFrame({k: pd.Series(phase[v]).value_counts() for k, v in SPLITS.items()}).fillna(0).astype(int)
    print('Days per ENSO phase in each split:'); display(oni_phase_counts)
    ph_test = phase[idx_test]
    rows = []
    for model, plist in all_preds_mm.items():
        for ph in np.unique(ph_test):
            m = ph_test == ph
            rows.append({'Model': model, 'ENSO phase': ph, 'n_days': int(m.sum()),
                         'RMSE (mm)': np.mean([rmse(y_test_mm[m], p[m]) for p in plist]),
                         'MAE (mm)': np.mean([mae(y_test_mm[m], p[m]) for p in plist])})
    error_by_enso = pd.DataFrame(rows)
    error_by_enso.to_csv(OUT_DIR / 'error_by_enso_phase.csv', index=False)
    oni_phase_counts.to_csv(OUT_DIR / 'enso_phase_counts_per_split.csv')
    display(error_by_enso.pivot_table(index='Model', columns='ENSO phase', values='RMSE (mm)', sort=False).round(3))
elif USE_ONI:
    print(f'USE_ONI=True but file not found: {ONI_PATH} — ENSO analysis skipped.')
else:
    print('USE_ONI=False — ENSO analysis skipped.')

## 8. Gambar untuk paper

Gambar 5 (baru): observed vs predicted pada periode **test saja**, enam panel dengan RMSE yang dihitung. Untuk model berbasis LSTM, panel menampilkan run yang RMSE-nya paling dekat dengan rata-rata seed (run representatif), dan judul memuat mean ± SD. Ditambah bar chart RMSE semua model dan baseline.

In [ ]:
def representative_run(name):
    runs = final_runs[name]
    i = int((runs.RMSE_mm - runs.RMSE_mm.mean()).abs().idxmin())
    return i, runs.loc[i]

PANEL_ORDER = ['Base SVR', 'W-SVR', 'Base LSTM', 'W-LSTM', 'Base LSTM-SVR', 'W-LSTM-SVR']
with plt.rc_context({'font.size': 14, 'axes.titlesize': 16, 'axes.labelsize': 14, 'legend.fontsize': 13}):
    fig, axes = plt.subplots(3, 2, figsize=(22, 16), sharex=True, sharey=True)
    for k, (ax, name) in enumerate(zip(axes.ravel(), PANEL_ORDER)):
        i, run = representative_run(name)
        r = results.set_index('Model').loc[name]
        title = f"({chr(97 + k)}) {name} — RMSE = {run.RMSE_mm:.3f} mm"
        if r['n_runs'] > 1:
            title += f"  [mean ± SD over {int(r['n_runs'])} seeds: {r['RMSE (mm)']:.3f} ± {r['RMSE SD (mm)']:.3f}]"
        ax.plot(test_dates, y_test_mm, color=C_OBS, lw=1.6, label='Observed')
        ax.plot(test_dates, final_preds_mm[name][i], color=C_WAVE, lw=1.6, alpha=0.9, label='Predicted')
        ax.set_title(title, loc='left')
        ax.grid(alpha=0.3)
        ax.xaxis.set_major_formatter(mdates.DateFormatter('%b %Y'))
        if k % 2 == 0: ax.set_ylabel('Rainfall (mm)')
        if k >= 4: ax.set_xlabel('Date')
        if k == 0: ax.legend(loc='upper left')
    fig.suptitle(f'Observed vs Predicted Daily Rainfall — Test Period {TEST_START} to {TEST_END} (n = {len(idx_test)} days)',
                 fontsize=18, y=1.0)
    fig.tight_layout()
    fig.savefig(OUT_DIR / 'fig5_observed_vs_predicted_test.png', dpi=300)
    fig.savefig(OUT_DIR / 'fig5_observed_vs_predicted_test.pdf')
    plt.show()

In [ ]:
bar = results[['Model', 'RMSE (mm)', 'RMSE SD (mm)']].copy()
colors = [C_WAVE if m.startswith('W-') else (C_BASE if m.startswith('Base') else C_GREY) for m in bar.Model]
fig, ax = plt.subplots(figsize=(12, 6))
ax.bar(bar.Model, bar['RMSE (mm)'], yerr=bar['RMSE SD (mm)'].fillna(0), color=colors, capsize=5,
       edgecolor='white', linewidth=2)
for x, (v, e) in enumerate(zip(bar['RMSE (mm)'], bar['RMSE SD (mm)'].fillna(0))):
    ax.text(x, v + e + 0.1, f'{v:.2f}', ha='center', va='bottom', fontsize=11, color=C_OBS)
ax.set_ylim(0, (bar['RMSE (mm)'] + bar['RMSE SD (mm)'].fillna(0)).max() * 1.15)
ax.set(ylabel='Test RMSE (mm)', title=f'Test RMSE of All Models and Baselines ({TEST_START} to {TEST_END})')
ax.tick_params(axis='x', rotation=30); ax.grid(axis='y', alpha=0.3)
from matplotlib.patches import Patch
ax.legend(handles=[Patch(color=C_BASE, label='Base'), Patch(color=C_WAVE, label='Wavelet (causal)'),
                   Patch(color=C_GREY, label='Baseline')], loc='upper left')
fig.savefig(OUT_DIR / 'fig_rmse_bar_all_models.png', dpi=300); plt.show()

## 9. Ringkasan untuk paper

Semua angka di bawah dihasilkan dari variabel di atas dan bisa langsung dikutip ke Abstract, Results, Tabel II, dan Conclusion.

In [ ]:
model_res = results[results.Model.isin(SCENARIOS)].set_index('Model')
best_model = model_res['RMSE (mm)'].idxmin()
b = model_res.loc[best_model]
red = reduction.set_index('Model family')['RMSE reduction (%)']
sp = split_tbl
lines = [
    '=' * 78, 'RINGKASAN UNTUK PAPER', '=' * 78,
    f'Mode: {"QUICK (uji cepat — JANGAN dipakai di paper)" if QUICK_MODE else "FULL"} | Optuna trials/scenario = {N_TRIALS} | '
    f'final seeds (LSTM-based) = {N_SEEDS_FINAL} | seed = {SEED}',
    '',
    '[Data]',
    f'  Period {df.index.min().date()} – {df.index.max().date()}, N = {len(df)} daily records '
    f'(missing dates = {n_missing_dates}, leading rows dropped = {n_dropped_head})',
    f'  Rainfall NaN after code conversion (8888/9999/TTU) = {int(missing_tbl.loc["Rainfall", "total NaN after conversion"])}; '
    f'imputation = forward fill only',
    f'  Days without rain = {PCT_DRY_DAYS:.2f}% | mean = {df[COL_RAIN].mean():.3f} mm, SD = {df[COL_RAIN].std():.3f} mm, '
    f'max = {df[COL_RAIN].max():.1f} mm',
    '',
    '[ADF test, daily rainfall]',
    f'  ADF statistic = {ADF_STAT:.4f}, p-value = {ADF_P:.3e}, lags = {ADF_USEDLAG}; critical values: '
    + ', '.join(f'{k} = {v:.4f}' for k, v in ADF_CRIT.items()),
    f'  Conclusion: {ADF_CONCLUSION}',
    f'  Significant ACF lags: {SIG_ACF[:15]}{" ..." if len(SIG_ACF) > 15 else ""} | significant PACF lags: {SIG_PACF[:15]}{" ..." if len(SIG_PACF) > 15 else ""}',
    '',
    '[Chronological split by target date]',
    *[f'  {k:<10}: {r.start} – {r.end} | n = {r.n_samples} | dry days = {r["dry days (%)"]}%' for k, r in sp.iterrows()],
    '',
    '[Test results — RMSE/MAE in mm after clipping negatives to 0]',
    *[f'  {r.Model:<22} ts = {"-" if pd.isna(r.time_steps) else int(r.time_steps)} | RMSE = {r["RMSE (mm) mean ± SD"]} | '
      f'MAE = {r["MAE (mm) mean ± SD"]} | RMSE scaled = {r["RMSE scaled mean ± SD"]}' for _, r in results.iterrows()],
    '',
    f'[Best model] {best_model}: RMSE = {b["RMSE (mm) mean ± SD"]} mm, MAE = {b["MAE (mm) mean ± SD"]} mm, '
    f'RMSE scaled = {b["RMSE scaled mean ± SD"]}',
    f'  Improvement vs persistence baseline (RMSE {PERSIST_RMSE:.3f} mm): {100 * (PERSIST_RMSE - b["RMSE (mm)"]) / PERSIST_RMSE:.2f}%',
    '',
    '[RMSE reduction from causal wavelet features = (RMSE_base - RMSE_wavelet) / RMSE_base x 100]',
    *[f'  {fam:<9}: {red[fam]:+.2f}%' for fam in red.index],
    '',
    '[Causality ablation — filtfilt is diagnostic only]',
    *[f'  {r.Model:<7} {r.Filter:<36} test RMSE = {r["Test RMSE (mm)"]:.3f} mm' for _, r in ablation_causality.iterrows()],
    '',
    f'[Error by day type] heavy-rain threshold = P90 of train rainfall = {HEAVY_THR:.2f} mm (see error_by_daytype.csv)',
    '=' * 78,
]
PAPER_SUMMARY = '\n'.join(lines)
print(PAPER_SUMMARY)
(OUT_DIR / 'paper_summary.txt').write_text(PAPER_SUMMARY)

print('\nOutput files:')
for f in sorted(OUT_DIR.rglob('*')):
    if f.is_file():
        print(f'  {f}  ({f.stat().st_size / 1024:.1f} KB)')